In [10]:
import subprocess
import sys
import multiprocessing
from kpis import calcular_kpis_promedio_con_ic_en_formato_json

def correr_sensibilidad_en_paralelo(
    seeds,
    script="run_sim_sensibilidad.py",
    info_cambio={"unidad": "ICU", "hospital": 2, "delta": +1}
):
    max_procesos = multiprocessing.cpu_count()
    procesos = []

    for i, seed in enumerate(seeds):
        print(f"🚀 Ejecutando seed={seed}")
        args = [
            sys.executable,
            script,
            str(seed),
            str(info_cambio["hospital"]),
            info_cambio["unidad"],
            str(info_cambio["delta"])
        ]
        p = subprocess.Popen(args)
        procesos.append(p)

        if len(procesos) >= max_procesos:
            for p in procesos:
                p.wait()
            procesos = []

    for p in procesos:
        p.wait()

    print("✅ Todas las simulaciones han terminado.")

In [15]:
import csv
import os
from datetime import datetime

def correr_sensibilidad_con_guardado_robusto():
    ruta_base = "resultados sensibilidad"
    os.makedirs(ruta_base, exist_ok=True)

    ruta_csv_exitosos = os.path.join(ruta_base, "resultados_sensibilidad_completados.csv")
    ruta_csv_fallidos = os.path.join(ruta_base, "resultados_sensibilidad_fallidos.csv")

    # Cargar progreso exitoso previo
    progreso_previos = set()
    if os.path.exists(ruta_csv_exitosos):
        with open(ruta_csv_exitosos, "r") as f:
            reader = csv.reader(f)
            next(reader)
            for row in reader:
                progreso_previos.add((int(row[0]), row[1], int(row[2])))

    unidades = ["OR", "ICU", "SDU/WARD"]
    hospitales = [1, 2, 3]
    seeds = list(range(1, 101))

    for h in hospitales:
        for u in unidades:
            delta = 1
            cambio_fue_marginal = False
            costo_promedio_anterior = 1e12

            while not cambio_fue_marginal:
                if (h, u, delta) in progreso_previos:
                    delta += 1
                    continue

                max_reintentos = 3
                intentos = 0
                exito = False
                razon_termino = ""

                while intentos < max_reintentos and not exito:
                    try:
                        folder_with_kpis = f"{ruta_base}/ModeloProactivo_T4500_C4208_H{h}_{u.replace('/','_')}/+{delta}/kpis"
                        info = {"unidad": u, "hospital": h, "delta": delta}
                        correr_sensibilidad_en_paralelo(seeds, info_cambio=info)

                        resumen = calcular_kpis_promedio_con_ic_en_formato_json(
                            folder_with_kpis, nivel_confianza=99, guardar_en=None
                        )
                        costo_str = resumen["costo_diario_promedio"]["General"]["total"]
                        costo_promedio = float(costo_str.split(" ± ")[0])

                        # Evaluar si se detiene
                        if (costo_promedio_anterior - costo_promedio < 1):
                            cambio_fue_marginal = True
                            razon_termino = "cambio marginal"
                        elif delta > 15:
                            cambio_fue_marginal = True
                            razon_termino = "delta > 15"
                        else:
                            costo_promedio_anterior = costo_promedio
                            razon_termino = ""

                        # Guardar resultado exitoso
                        with open(ruta_csv_exitosos, "a", newline="") as f:
                            writer = csv.writer(f)
                            if os.stat(ruta_csv_exitosos).st_size == 0:
                                writer.writerow(["hospital", "unidad", "delta", "costo", "timestamp", "termino"])
                            writer.writerow([h, u, delta, costo_promedio, datetime.now().isoformat(), razon_termino])

                        progreso_previos.add((h, u, delta))
                        exito = True

                        # Solo aumentar delta si no se detuvo
                        if not cambio_fue_marginal:
                            delta += 1

                    except Exception as e:
                        intentos += 1
                        if intentos >= max_reintentos:
                            with open(ruta_csv_fallidos, "a", newline="") as f:
                                writer = csv.writer(f)
                                if os.stat(ruta_csv_fallidos).st_size == 0:
                                    writer.writerow(["hospital", "unidad", "delta", "timestamp", "comentario"])
                                writer.writerow([h, u, delta, datetime.now().isoformat(), "Error tras múltiples intentos"])
                            delta += 1

In [ ]:
correr_sensibilidad_con_guardado_robusto()

In [20]:
import csv
import os
from datetime import datetime

def correr_sensibilidad_con_guardado_robusto(max_delta=15):
    """
    Ejecuta análisis de sensibilidad incremental en camas por unidad y hospital.

    Parámetros:
    - max_delta: límite máximo de incrementos delta a probar por combinación (hospital, unidad)
    """
    ruta_base = "resultados sensibilidad"
    os.makedirs(ruta_base, exist_ok=True)

    ruta_csv_exitosos = os.path.join(ruta_base, "resultados_sensibilidad_completados.csv")
    ruta_csv_fallidos = os.path.join(ruta_base, "resultados_sensibilidad_fallidos.csv")

    # Cargar resultados ya completados
    progreso_previos = set()
    pares_ya_finalizados = set()

    if os.path.exists(ruta_csv_exitosos):
        with open(ruta_csv_exitosos, "r") as f:
            reader = csv.DictReader(f)
            for row in reader:
                h = int(row["hospital"])
                u = row["unidad"]
                d = int(row["delta"])
                t = row["termino"].strip()
                progreso_previos.add((h, u, d))

                if t in ["cambio marginal", f"delta > {max_delta}"]:
                    pares_ya_finalizados.add((h, u))

    unidades = ["OR", "ICU", "SDU/WARD"]
    hospitales = [1, 2, 3]
    seeds = list(range(1, 101))

    for h in hospitales:
        for u in unidades:
            if (h, u) in pares_ya_finalizados:
                continue  # Saltar unidad ya finalizada

            delta = 1
            cambio_fue_marginal = False
            costo_promedio_anterior = 1e12

            while not cambio_fue_marginal:
                if (h, u, delta) in progreso_previos:
                    delta += 1
                    continue
                print(f"🛠️  Trabajando en sensibilidad: Hospital {h}, Unidad {u}, comenzando en delta {delta}")

                max_reintentos = 3
                intentos = 0
                exito = False
                razon_termino = ""

                while intentos < max_reintentos and not exito:
                    try:
                        folder_with_kpis = f"{ruta_base}/ModeloProactivo_T4500_C4208_H{h}_{u.replace('/','_')}/+{delta}/kpis"
                        info = {"unidad": u, "hospital": h, "delta": delta}
                        correr_sensibilidad_en_paralelo(seeds, info_cambio=info)

                        resumen = calcular_kpis_promedio_con_ic_en_formato_json(
                            folder_with_kpis, nivel_confianza=99, guardar_en=None
                        )
                        costo_str = resumen["costo_diario_promedio"]["General"]["total"]
                        costo_promedio = float(costo_str.split(" ± ")[0])

                        # Evaluar si se detiene
                        if (costo_promedio_anterior - costo_promedio < 1):
                            cambio_fue_marginal = True
                            razon_termino = "cambio marginal"
                            pares_ya_finalizados.add((h, u))
                        elif delta > max_delta:
                            cambio_fue_marginal = True
                            razon_termino = f"delta > {max_delta}"
                            pares_ya_finalizados.add((h, u))
                        else:
                            costo_promedio_anterior = costo_promedio
                            razon_termino = ""

                        # Guardar resultado exitoso
                        with open(ruta_csv_exitosos, "a", newline="") as f:
                            writer = csv.writer(f)
                            if os.stat(ruta_csv_exitosos).st_size == 0:
                                writer.writerow(["hospital", "unidad", "delta", "costo", "timestamp", "termino"])
                            writer.writerow([h, u, delta, costo_promedio, datetime.now().isoformat(), razon_termino])

                        progreso_previos.add((h, u, delta))
                        exito = True

                        if not cambio_fue_marginal:
                            delta += 1

                    except Exception as e:
                        intentos += 1
                        if intentos >= max_reintentos:
                            with open(ruta_csv_fallidos, "a", newline="") as f:
                                writer = csv.writer(f)
                                if os.stat(ruta_csv_fallidos).st_size == 0:
                                    writer.writerow(["hospital", "unidad", "delta", "timestamp", "comentario"])
                                writer.writerow([h, u, delta, datetime.now().isoformat(), "Error tras múltiples intentos"])
                            delta += 1

In [ ]:
correr_sensibilidad_con_guardado_robusto()